# Tutorial 6 — Artifact studies, surveys and caching

**Goal:** run the kind of systematic study the package was built for — sweep one artifact,
survey all of them, and save everything to disk so the analysis can be redone without
re-simulating.

**You will learn:** `run`, `run_batch`, `signature_table`, `run_artifact_survey`,
`compare_algorithms`, and `SimCache`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import neutron_xray_sim as nxs

print("DIANA / neutron_xray_sim", nxs.__version__)

import tempfile, pathlib
workdir = pathlib.Path(tempfile.mkdtemp(prefix="diana_tutorial_"))
print("results will be cached in", workdir)

## 1. A parameter sweep: misalignment

Dual-modality data usually comes from two separate scans, which must be registered. How
precise does that registration need to be? Sweep the residual shift and watch the clusters
smear. The raw sinograms are projected once and reused for every run.

In [ ]:
sim = nxs.DualModalitySimulation(preset="composite", N=48, n_angles=90, verbose=False,
                                 cache_dir=workdir / "misalignment")
ref = sim.run(tag="0 vx")
shifts = [0.5, 1, 2, 4]
runs = [ref] + [sim.run(nxs.ArtifactConfig(misalignment=True, translation_voxels=(0, d, 0)),
                        tag=f"{d} vx", ref_result=ref) for d in shifts]
fig = sim.comparison_grid(runs, ncols=5)

We score each run with the label-anchored metrics from Tutorial 5. They are more stable
than GMM-based metrics, whose component placement can vary from run to run.

In [ ]:
def label_metrics(r, phantom):
    table, _ = nxs.compute_histogram_metrics_morphology_aware(
        phantom, r.histogram, r.vol_xray, r.vol_neutron)
    return table.scalars["CE"], table.scalars["DB"]

ce, db = zip(*[label_metrics(r, sim.phantom) for r in runs])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot([0] + shifts, ce, "o-"); ax[0].set_ylabel("mean centroid error [cm⁻¹]")
ax[1].plot([0] + shifts, db, "o-"); ax[1].set_ylabel("Davies–Bouldin index")
for a in ax: a.set_xlabel("misalignment [voxels]"); a.grid(alpha=.3)
print(sim.signature_table())

## 2. Everything is on disk

Because we passed `cache_dir`, the phantom, the raw sinograms, and every run's
volumes/histograms are saved as `.npy` + JSON. A later session (or a colleague) can reload
them without recomputing.

In [ ]:
cache = nxs.SimCache(workdir / "misalignment")
print(cache)
print("runs:", cache.list_run_tags())
vol_x = cache.load_run_volume("2 vx", "xray")
vol_n = cache.load_run_volume("2 vx", "neutron")
phantom = cache.load_phantom()
h = nxs.compute_bimodal_histogram(vol_x, vol_n, bins=128)
table, _ = nxs.compute_histogram_metrics_morphology_aware(phantom, h, vol_x, vol_n)
print("re-analysed from disk: CE =", round(table.scalars["CE"], 4))

If you change the projection settings (e.g. `n_angles`) but reuse the same `cache_dir`,
DIANA notices and refuses to reuse stale sinograms unless you pass `overwrite_cache=True`.

## 3. The artifact survey

`run_artifact_survey` runs a clean reference, each artifact on its own, and some combinations,
then draws one panel per run with ground-truth markers and metrics. This is the fastest way
to get a feel for a new phantom (about a minute at this size).

In [ ]:
results, fig, survey_metrics = nxs.run_artifact_survey(
    preset="composite", N=40, n_angles=60, histogram_bins=96,
    include_combinations=False, verbose=False,
)

The panel titles show the survey's GMM-based metrics. To rank the artifacts, the
label-anchored CE is more robust:

In [ ]:
phantom = next(iter(results.values())).phantom
scores = {tag: label_metrics(r, phantom) for tag, r in results.items()}
for tag, (ce_, db_) in sorted(scores.items(), key=lambda kv: kv[1][0]):
    print(f"{tag.replace(chr(10), ' '):<40} CE={ce_:.3f}  DB={db_:.3f}")

**Something to think about:** X-ray scatter and beam-hardening correction *lower* the
centroid error here. Look at the per-material ε_k of the iron: beam hardening pushes the metal
clusters away from their 80 keV reference, and these two effects happen to push them back. A
single summary number can hide compensating errors — always look at the per-material values.

**Exercise:** repeat the survey for the `battery` preset. Which artifact hurts the
electrolyte/separator separation most — and would a better neutron detector (PSF) or longer
exposure (I₀) help more?